In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error , r2_score

In [2]:
df = pd.read_csv('../data/features/features_complete.csv')


features = [

    # --- Title / NLP features ---
    'title_length',
    'title_word_count',
    'uppercase_words',
    'num_emojis',
    'has_emoji',
    'contains_numbers_or_emojis',
    'has_question',
    'is_clickbait',
    'sentiment_polarity',
    'sentiment_subjectivity',

    # ---  content ---
    'top50_pca1',
    'top50_pca2',
    'top50_pca3',

    # --- Time Features ---
    'is_published_weekend',

    # --- Metadata ---
    'category_id',
    'comments_disabled',
    'ratings_disabled',


     'days_until_trending',     
]
X = df[features]
y = np.log1p(df['views'])
print(f"Shape: X={X.shape}, y={y.shape}")

Shape: X=(55886, 18), y=(55886,)


In [3]:
lr = LinearRegression()
X_train, X_test, y_train, y_test  = train_test_split(X,y,test_size=0.2, random_state=42)
lr.fit(X_train, y_train)


LinearRegression()

In [4]:
y_pred = lr.predict(X_test)

In [5]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\nLinear Regression Model Performance (LOG SCALE):")
print(f"RMSE: {rmse:.4f}")  # ← Changed from :.0f to :.4f
print(f"MAE:  {mae:.4f}")   # ← Changed from :.0f to :.4f
print(f"R²:   {r2:.4f}")


Linear Regression Model Performance (LOG SCALE):
RMSE: 1.5593
MAE:  1.2762
R²:   0.1367


In [6]:
from sklearn.preprocessing import StandardScaler

# Scale features (required for Ridge/Lasso)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled!")

Features scaled!


In [7]:
alphas = [0.01, 0.1, 1, 10, 100]

print("\n" + "="*60)
print("RIDGE REGRESSION RESULTS")
print("="*60)

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)
    y_pred_ridge = ridge.predict(X_test_scaled)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
    mae = mean_absolute_error(y_test, y_pred_ridge)
    r2 = r2_score(y_test, y_pred_ridge)
    
    print(f"Ridge (α={alpha:>6.2f}): RMSE={rmse:.4f}, MAE={mae:.4f}, R²={r2:.4f}")


RIDGE REGRESSION RESULTS
Ridge (α=  0.01): RMSE=1.5593, MAE=1.2762, R²=0.1367
Ridge (α=  0.10): RMSE=1.5593, MAE=1.2762, R²=0.1367
Ridge (α=  1.00): RMSE=1.5593, MAE=1.2762, R²=0.1367
Ridge (α= 10.00): RMSE=1.5593, MAE=1.2762, R²=0.1367
Ridge (α=100.00): RMSE=1.5593, MAE=1.2763, R²=0.1367


In [8]:
alphas = [0.01, 0.1, 1, 10, 100]

print("\n" + "="*60)
print("Lasso REGRESSION RESULTS")
print("="*60)

for alpha in alphas:
    lasso = Lasso(alpha=alpha)
    lasso.fit(X_train_scaled, y_train)
    y_pred_lasso = lasso.predict(X_test_scaled)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
    mae = mean_absolute_error(y_test, y_pred_lasso)
    r2 = r2_score(y_test, y_pred_lasso)
    
    print(f"Lasso (α={alpha:>6.2f}): RMSE={rmse:.4f}, MAE={mae:.4f}, R²={r2:.4f}")


Lasso REGRESSION RESULTS
Lasso (α=  0.01): RMSE=1.5598, MAE=1.2779, R²=0.1361
Lasso (α=  0.10): RMSE=1.5898, MAE=1.3105, R²=0.1026
Lasso (α=  1.00): RMSE=1.6782, MAE=1.3909, R²=-0.0000
Lasso (α= 10.00): RMSE=1.6782, MAE=1.3909, R²=-0.0000
Lasso (α=100.00): RMSE=1.6782, MAE=1.3909, R²=-0.0000


In [9]:
print("\n" + "="*70)
print("FEATURE SELECTION BY LASSO")
print("="*70)

for alpha in [0.01, 0.1, 1.0]:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train_scaled, y_train)
    
    n_features = np.sum(np.abs(lasso.coef_) > 1e-5)
    
    print(f"\n{'='*70}")
    print(f"Lasso (α={alpha}) - Using {n_features}/15 features")
    print(f"{'='*70}")
    
    # Sort by absolute coefficient value
    feature_importance = [(feat, coef) for feat, coef in zip(features, lasso.coef_)]
    feature_importance.sort(key=lambda x: abs(x[1]), reverse=True)
    
    print("\nKept Features (sorted by importance):")
    for feat, coef in feature_importance:
        if abs(coef) > 1e-5:
            print(f"  ✓ {feat:<30}: {coef:>8.4f}")
    
    print("\nDropped Features:")
    dropped = [feat for feat, coef in feature_importance if abs(coef) <= 1e-5]
    if dropped:
        for feat in dropped:
            print(f"  ✗ {feat}")
    else:
        print("  (none)")


FEATURE SELECTION BY LASSO

Lasso (α=0.01) - Using 16/15 features

Kept Features (sorted by importance):
  ✓ top50_pca2                    :  -0.3607
  ✓ has_emoji                     :  -0.2710
  ✓ title_length                  :   0.2448
  ✓ sentiment_subjectivity        :   0.2051
  ✓ uppercase_words               :  -0.1463
  ✓ comments_disabled             :  -0.0894
  ✓ num_emojis                    :   0.0863
  ✓ contains_numbers_or_emojis    :  -0.0845
  ✓ ratings_disabled              :  -0.0628
  ✓ has_question                  :  -0.0515
  ✓ title_word_count              :  -0.0393
  ✓ days_until_trending           :  -0.0264
  ✓ sentiment_polarity            :  -0.0044
  ✓ is_clickbait                  :   0.0015
  ✓ category_id                   :   0.0015
  ✓ is_published_weekend          :   0.0007

Dropped Features:
  ✗ top50_pca1
  ✗ top50_pca3

Lasso (α=0.1) - Using 7/15 features

Kept Features (sorted by importance):
  ✓ top50_pca2                    :  -0.2859
  ✓ 